<a href="https://colab.research.google.com/github/ShinAsakawa/ShinAsakawa.github.io/blob/master/2026notebooks/2026_0715MDD_Spacy_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spacy, GiNZA を用いた平均依存距離の算出

    - Author: 浅川 伸一 asakawa@ieee.org
    - Date: 2026_0715


* [spaCy](https://spacy.io/)

spaCy は高度な自然言語処理を行うためのライブラリ。

自然言語処理では対象とする言語（日本語や英語）によって必要な処理や複雑度が変わるのですが、spaCy は多言語対応を意識して設計・開発されいる。

* [GiNZA](https://megagonlabs.github.io/ginza/)

GiNZAは日本語の自然言語処理を行うためのライブラリでリクルートと国語研が共同で開発したライブラリ。
GiNZA は spaCy の API を使用して学習されており、spaCy からモデルをロードして使用することができる。


* [統語依存関係に基づく位相研究, 李 他 2022](https://www.ls-japan.org/modules/documents/LSJpapers/journals/162_li.pdf)


* [日本語 Universal Dependencies の試案](https://www.anlp.jp/proceedings/annual_meeting/2015/pdf_dir/E3-4.pdf)

* [spaCyとGiNZAを使った日本語自然言語処理](https://qiita.com/wf-yamaday/items/3ffdcc15a5878b279d61)

In [2]:
# GiNZA のインストール
try:
    import ginza
except ImportError:
    !pip install -U ginza ja_ginza

In [3]:
import spacy

# GiNZAモデルの読み込み
# Python 3.12 + Spacy 3.8 + Ginza 5.2 の構成だとそのままでは動作しないため 以下の設定を追加指定
config = {
    "components": {
        "compound_splitter": {
            "split_mode": "A"
        }
    }
}
nlp = spacy.load("ja_ginza", config=config)

# テストするテキスト
text = "二事業の指定取り消しを決めた"
doc = nlp(text)

# 依存距離の計算
total_dist = 0
count = 0

for token in doc:
    if token.dep_ != "ROOT":
        dist = abs(token.head.i - token.i)
        total_dist += dist
        count += 1

mdd = total_dist / count if count > 0 else 0

print(f"解析テキスト: {text}")
print(f"単語単位の平均依存距離 (MDD): {mdd:.2f}")


解析テキスト: 二事業の指定取り消しを決めた
単語単位の平均依存距離 (MDD): 2.14


In [13]:
for token in doc:
    if token.dep_ != "ROOT":
        dist = abs(token.head.i - token.i)
        print(token, token.head.i, token.i, token.head.i - token.i)


二 6 0 6
事業 4 1 3
の 1 2 -1
指定 4 3 1
取り消し 6 4 2
を 4 5 -1
た 6 7 -1


In [22]:
def mdd(doc):
    doc = nlp(text)

    # 依存距離の計算
    total_dist = 0
    count = 0

    for token in doc:
        if token.dep_ != "ROOT":
            dist = abs(token.head.i - token.i)
            total_dist += dist
            count += 1

    _mdd = total_dist / count if count > 0 else 0

    #print(f"解析テキスト: {text}")
    print(f"単語単位の平均依存距離 (MDD): {_mdd:.2f}")
    return _mdd


In [23]:
sentences = ['クロールで泳いでいる少女を見た', '望遠鏡で泳いでいる少女を見た', '機内の楽しみって「おやつ」と「機内食」しかなく〜今まで満足したことなかったけど、フィリピンエアライン〜良かったよ！']
for s in sentences:
    print(s, f'mdd:{mdd(s):.2f}')


単語単位の平均依存距離 (MDD): 2.14
クロールで泳いでいる少女を見た mdd:2.14
単語単位の平均依存距離 (MDD): 2.14
望遠鏡で泳いでいる少女を見た mdd:2.14
単語単位の平均依存距離 (MDD): 2.14
機内の楽しみって「おやつ」と「機内食」しかなく〜今まで満足したことなかったけど、フィリピンエアライン〜良かったよ！ mdd:2.14


In [24]:
# text を Doc クラスに変換する
text: str = '錦織圭選手はテニスが大好きです。'
doc: spacy.tokens.doc.Doc = nlp(text)

# Doc クラスは Token クラスのイテレーターになっている
for token in doc:
  print(token.text, type(token)) # token.text は日本語の形態素の単位


錦織 <class 'spacy.tokens.token.Token'>
圭 <class 'spacy.tokens.token.Token'>
選手 <class 'spacy.tokens.token.Token'>
は <class 'spacy.tokens.token.Token'>
テニス <class 'spacy.tokens.token.Token'>
が <class 'spacy.tokens.token.Token'>
大好き <class 'spacy.tokens.token.Token'>
です <class 'spacy.tokens.token.Token'>
。 <class 'spacy.tokens.token.Token'>


In [25]:
from spacy import displacy

# 依存関係の可視化（jupyter=TrueとすることでNotebook上で表示できる）
displacy.render(doc, style="dep", options={"compact":True},  jupyter=True)


In [26]:
# エンティティの可視化（jupyter=True とすることで Notebook 上で表示できる）
displacy.render(doc, style="ent", options={"compact":True},  jupyter=True)


In [27]:
doc2 = nlp('錦織圭選手は偉大なテニス選手です。')

# noun_chunksでテキスト文に含まれる名詞句を取り出す
for chunk in doc2.noun_chunks:
  print(chunk.text, type(chunk))


錦織圭選手 <class 'spacy.tokens.span.Span'>
偉大なテニス選手 <class 'spacy.tokens.span.Span'>


In [28]:
# 品詞タグから名詞の単語を抽出する
for token in doc2:
  if token.pos_ in ['NOUN', 'PROPN']: # NOUNが名詞、PROPNが固有名詞
    print(token.text, token.tag_, type(token))


錦織 名詞-固有名詞-人名-姓 <class 'spacy.tokens.token.Token'>
圭 名詞-固有名詞-人名-名 <class 'spacy.tokens.token.Token'>
選手 名詞-普通名詞-一般 <class 'spacy.tokens.token.Token'>
テニス 名詞-普通名詞-一般 <class 'spacy.tokens.token.Token'>
選手 名詞-普通名詞-一般 <class 'spacy.tokens.token.Token'>


In [29]:
print('doc1',doc.text)
print('doc2',doc2.text)

# 2つの文の類似度を求める
print('cos類似度:', doc.similarity(doc2))


doc1 錦織圭選手はテニスが大好きです。
doc2 錦織圭選手は偉大なテニス選手です。
cos類似度: 0.9598305821418762


In [30]:
text3 = '''
　阪神がリーグ優勝を逃した。すでに勝利で試合を終えたヤクルトがマジック１としており、この試合に敗れた瞬間、ヤクルトの６年ぶりの優勝と阪神のＶ逸が決定。阪神にとって１６年ぶりの夢が、本拠地で散った。
　ミスで無駄な点を与える、今季を象徴するような戦いぶりだった。二回１死一、二塁、木下拓の三ゴロで併殺コースは、二塁手・糸原が一塁へ悪送球する適時失策で先制点を献上した。
　０－１の五回は無死から２番手・及川が、先頭・岡林をスライダーで空振り三振に仕留めたが、ワンバウンドした球を捕手・坂本が一塁ベンチ方向にそらし、振り逃げで出塁を許した。（記録は投手の暴投）この後、四球、安打で１死満塁として三番手・馬場に交代。馬場は２死後、大島に２点適時打を浴びた。
　負けられない１戦。矢野監督は二回、２死一、三塁の好機に、先発の青柳に代打・小野寺を送る積極采配。結果は遊飛に終わった。決死のリレーも及川が３イニング目につかまった。今季最終戦だが、“第２先発”を任せられる他の先発陣をベンチにはスタンバイさせる策を取っていなかった。
　打線も沈黙した。糸原が猛打賞と気を吐いたが、その他は大山の１安打のみ。計４安打完封負けで今季を終えた。
　今季は佐藤輝、中野、伊藤将のルーキートリオが大活躍。マルテ、スアレスら外国人も機能し、５月まで破竹の勢いで白星を重ねた。一時は２位に最大７ゲーム差をつけ、独走の雰囲気も漂った。
　だが、打線の勢いが低下し、夏場に失速。８月末に首位の座を奪われた。１０月は投手陣の奮闘もあり、ヤクルトに負けじと貯金を増やした。最後までＶへの執念もみせたが、わずかに頂点へ届かなかった。
　矢野監督は試合後の挨拶で「今日のこの最後の試合、こういう試合で勝ちきれる、もっともっといいチームに、もっともっと強いチームになっていけるよう。新たなスタートとして、この悔しさを持って戦っていきます」と今後に向けて語った。
'''

doc3 = nlp(text3)

# 重要部分をresultsに保存する
results = []
for chunk in doc3.noun_chunks:
    results.append((chunk.text, chunk.similarity(doc3)))

# 上位10個の名詞句を表示する
print(sorted(results,key=lambda x: x[1],reverse=True)[:10])


[('無駄な点を与える、今季を象徴するような戦いぶり', 0.8739744424819946), ('試合を終えたヤクルト', 0.8052458167076111), ('ワンバウンドした球を捕手・坂本', 0.7959782481193542), ('代打・小野寺を送る積極采配', 0.7768658399581909), ('新たなスタート', 0.7054125666618347), ('もっともっと強いチーム', 0.7006116509437561), ('「今日', 0.6996909976005554), ('こういう試合', 0.6925894618034363), ('三番手・馬場', 0.6916490793228149), ('もっともっといいチーム', 0.6826406121253967)]


/tmp/ipykernel_3551/3943218720.py:17: UserWarning: [W008] Evaluating Span.similarity based on empty vectors.
  results.append((chunk.text, chunk.similarity(doc3)))
